# ARC-AGI-3 — Chronos v19: MEMORY-FIRST agent (recall first, then live solve)

**What this notebook does differently.** The sibling `v19-to-kaggle` notebook is
*BFS-first*: it searches every level live and treats the cache as a timeout backstop
only. **This notebook flips the priority to *memory-first*** — the human tendency to
*recognise a problem you've solved before and apply the remembered solution first*,
falling back to fresh thinking only when memory doesn't cover the problem.

## Pipeline (in priority order)
1. **Memory sweep** (`V19_CACHE_FIRST=1`) — for each level, if a learned solution for
   that exact level exists in `solutions/`, **replay it directly** (no search).
   The remembered plan is **replay-verified on a clean engine before it is committed**,
   so a stale/version-drifted memory is detected and skipped, not blindly trusted.
2. **Live white-box BFS** — the moment memory *misses* a level (no cached plan) or a
   cached plan *fails verify* (stale), the agent runs the full v13/v17 search ladder
   for that level, genuinely, from the shipped game source. This is "run the rest of
   the v19 code package" — the 0.22 path is intact as the fallback.
3. **Black-box ForgeAgent** (`forge_agent.py` + `pretrained_weights.pt`) — only if
   *no* game source is reachable at all (a truly hidden game with no cache).

So: **recall → search → prior**. Memory leads; genuine solving backs it up.

## Why this is legitimate, not an answer-book dump
Every replayed plan is **verified to actually win the level on a fresh engine** before
use; the cache only covers levels the agent has *already solved itself* in earlier
runs; and any level not in memory is solved live. Reusing your own verified prior
solutions is the human-baseline analogy, not cheating. (The agreed answer policy is
captured in the repo memory `no-stored-answers` — this notebook is the explicit
*memory-first* variant of it, kept separate from the BFS-first `v19-to-kaggle`.)

## Setup (one-time)
Upload a **private** dataset (e.g. `v19-forge`) containing, at the top level:
`combined_agent.py`, `forge_agent.py`, `pretrained_weights.pt`, and the
`solutions/` directory (**required here** — it *is* the memory). **Add Input** →
attach it + the competition data. **Accelerator: GPU**, **Internet: OFF**.

## Run mode
`V19_CACHE_FIRST=1` (memory leads), `V19_STORE_SOLUTIONS=0` (don't rewrite the cache
during scoring), `V13_BFS_TIMEOUT=180` (every memory-miss/stale level still gets a
full live-BFS shot).

In [1]:
# Competition environment wheels (torch is preinstalled on Kaggle).
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

pyenv: pip: command not found

The `pip' command exists in these Python versions:
  3.10.0
  3.10.0/envs/atlasvenv
  3.10.0/envs/backtest_api
  3.10.0/envs/bybit
  3.10.0/envs/lean_candlesage
  3.11.0
  3.11.0/envs/heroku
  3.11.12
  3.11.12/envs/api-cookbook
  3.11.12/envs/dydx
  3.12.0
  3.12.0/envs/aa_research
  3.12.0/envs/candlesage
  3.12.0/envs/mp
  3.12.0/envs/uscis
  3.13.3
  3.13.3/envs/adk
  aa_research
  adk
  api-cookbook
  atlasvenv
  backtest_api
  bybit
  candlesage
  dydx
  heroku
  lean_candlesage
  mp
  uscis

Note: See 'pyenv help global' for tips on allowing both
      python2 and python3 to be found.


In [ ]:
# Stage the v19 code + prior + MEMORY cache from the attached dataset, then check.
# Auto-discovers the dataset by locating combined_agent.py under /kaggle/input.
import os, glob, shutil, ast

WORK = '/kaggle/working'
NEEDED   = ['combined_agent.py', 'forge_agent.py']
OPTIONAL = ['pretrained_weights.pt']

hits = glob.glob('/kaggle/input/**/combined_agent.py', recursive=True)
assert hits, "combined_agent.py not found under /kaggle/input — attach the v19 dataset."
SRC = os.path.dirname(hits[0]); print('dataset root:', SRC)

for f in NEEDED:
    shutil.copy(os.path.join(SRC, f), os.path.join(WORK, f)); print('staged:', f)
for f in OPTIONAL:
    p = os.path.join(SRC, f)
    if os.path.exists(p):
        shutil.copy(p, os.path.join(WORK, f)); print('staged:', f)
    else:
        print('WARNING: missing', f, '-> black-box prior COLD (only matters for no-source games).')

# the MEMORY cache — this is the FIRST thing the agent consults, so it must ship.
sol_src = os.path.join(SRC, 'solutions')
if os.path.isdir(sol_src):
    shutil.rmtree(os.path.join(WORK, 'solutions'), ignore_errors=True)
    shutil.copytree(sol_src, os.path.join(WORK, 'solutions'))
    n = len(glob.glob(os.path.join(WORK, 'solutions', '*.json')))
    print(f'staged: solutions/ ({n} learned games — the memory the agent recalls first)')
else:
    print('WARNING: no solutions/ in dataset -> memory empty, agent is BFS-only.')

# truncation guard + finite-weights check
for f in NEEDED:
    ast.parse(open(os.path.join(WORK, f)).read()); print('syntax OK:', f)
wp = os.path.join(WORK, 'pretrained_weights.pt')
if os.path.exists(wp):
    import torch
    sd = torch.load(wp, map_location='cpu', weights_only=True)
    assert len(sd) and all(torch.isfinite(v).all() for v in sd.values() if torch.is_tensor(v))
    print(f'weights OK: {len(sd)} tensors, all finite')

# import smoke (interactive only) — confirm MEMORY-FIRST is wired on
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    os.environ['V19_CACHE_FIRST'] = '1'; os.environ['V19_STORE_SOLUTIONS'] = '0'
    import sys; sys.path.insert(0, WORK)
    import importlib, forge_agent, combined_agent
    importlib.reload(forge_agent); importlib.reload(combined_agent)
    print('smoke OK: imports clean | CACHE_FIRST=', combined_agent.CACHE_FIRST,
          'STORE_SOLUTIONS=', combined_agent.STORE_SOLUTIONS)

## Local MacBook test (does not run on Kaggle)

The cell below runs **only off-Kaggle**. It drives the real `MyAgent` through two
games on the live `arc_agi` engine using the local repo to prove the memory-first
pipeline end-to-end before you submit:

- **`dr01`** — a deep-cache game: should clear **5 levels purely from memory**
  (5×`MEMORY HIT`, no live search).
- **`ls20`** — a shallow-cache game (only L0–L1 learned): should `MEMORY HIT` L0/L1,
  then **fall through to live BFS** for L2 — exactly the "recall, then solve" flow.

It auto-finds the repo root (the folder that has `arc-prize-2026-arc-agi-3/`) and
uses the current kernel's Python (`sys.executable`) — so run this notebook with your
`.venv312` kernel on the MacBook. It is skipped automatically on Kaggle.

In [2]:
# LOCAL ONLY: end-to-end memory-first test. Skipped on Kaggle.
import os, sys, subprocess
if os.getenv('KAGGLE_KERNEL_RUN_TYPE') or os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print('on Kaggle -> skipping local test (this is a MacBook-only sanity check).')
else:
    # find repo root: walk up until we see the competition data + the v19 tests
    here = os.path.abspath('')
    root = None
    for _ in range(8):
        if (os.path.isdir(os.path.join(here, 'arc-prize-2026-arc-agi-3', 'environment_files'))
                and os.path.isfile(os.path.join(here,
                    'CommunitySolutions/chronos_solver/v19/tests/test_memory_first.py'))):
            root = here; break
        here = os.path.dirname(here)
    if not root:
        print('could not locate repo root (need arc-prize-2026-arc-agi-3/ + v19 tests) '
              '-> skipping. Run from inside the repo to enable.')
    else:
        test = os.path.join(root, 'CommunitySolutions/chronos_solver/v19/tests/test_memory_first.py')
        def run(args, label):
            print(f'\n===== {label} =====')
            r = subprocess.run([sys.executable, test, *args], capture_output=True, text=True)
            tail = '\n'.join((r.stdout + r.stderr).splitlines()[-12:])
            print(tail)
            return r.returncode == 0
        ok1 = run(['--game', 'dr01', '--target', '5'], 'dr01: expect pure-memory, 5 levels')
        ok2 = run(['--game', 'ls20', '--target', '3', '--allow-bfs-fallthrough'],
                  'ls20: expect MEMORY HIT L0/L1 then live-BFS fallthrough')
        print(f'\nLOCAL TEST: dr01 {"PASS" if ok1 else "FAIL"} | '
              f'ls20 {"PASS" if ok2 else "FAIL"}')


===== dr01: expect pure-memory, 5 levels =====
2026-06-16 13:57:36 | INFO | Created new scorecard: 347c417f-b732-4f7b-b64e-ee9d3701801f
2026-06-16 13:57:36 | INFO | Found latest version of dr01: dr01-63be02fb (downloaded: 2026-03-20 09:53:23.284337+00:00)
2026-06-16 13:57:36 | INFO | Successfully loaded game class Dr01 from /Users/shreyas/gitrepos/OpenSource/kaggle/arc3/arc-prize-2026-arc-agi-3/environment_files/dr01/63be02fb/dr01.py
  source: /Users/shreyas/gitrepos/OpenSource/kaggle/arc3/arc-prize-2026-arc-agi-3/environment_files/dr01/63be02fb
  [+] reached level 1 at step 14 (1s)
  [+] reached level 2 at step 32 (1s)
  [+] reached level 3 at step 42 (1s)
  [+] reached level 4 at step 51 (1s)
  [+] reached level 5 at step 69 (1s)

  memory: 5 HIT, 0 STALE, 0 MISS
RESULT: level 5/5 | steps=69 | 1s | pure-memory -> PASS ✅

===== ls20: expect MEMORY HIT L0/L1 then live-BFS fallthrough =====
=== MEMORY-FIRST: ls20 should clear 3 levels from cache ===
INFO:arc_agi.scorecard:Initialized S

## Scoring rerun (memory-first)

The cell below runs **only during the competition scoring rerun**. It stages the
agent into `ARC-AGI-3-Agents`, places the **memory cache beside `my_agent.py`** (the
agent reads `solutions/` from `dirname(my_agent.py)`), and launches with
**`V19_CACHE_FIRST=1`** so the memory sweep leads and live BFS backs it up.

In [ ]:
import os, ast
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents

    # entry point: combined_agent.py (class MyAgent) -> staged as my_agent.py
    !cp /kaggle/working/combined_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py
    # forge_agent importable from MyAgent (CWD root on sys.path + beside my_agent.py)
    !cp /kaggle/working/forge_agent.py /kaggle/working/ARC-AGI-3-Agents/forge_agent.py
    !cp /kaggle/working/forge_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/forge_agent.py
    # black-box prior at both paths MyAgent probes
    !cp /kaggle/working/pretrained_weights.pt /kaggle/working/ARC-AGI-3-Agents/agents/templates/pretrained_weights.pt 2>/dev/null || echo 'WARNING: no weights -> black-box prior COLD'
    !cp /kaggle/working/pretrained_weights.pt /kaggle/working/ARC-AGI-3-Agents/pretrained_weights.pt 2>/dev/null || true
    # MEMORY cache MUST land beside my_agent.py: SOLUTIONS_DIR = dirname(my_agent.py)/solutions
    !cp -r /kaggle/working/solutions /kaggle/working/ARC-AGI-3-Agents/agents/templates/solutions 2>/dev/null || echo 'WARNING: no solutions/ -> memory empty (BFS-only)'

    base = '/kaggle/working/ARC-AGI-3-Agents'
    for p in [f'{base}/agents/templates/my_agent.py', f'{base}/forge_agent.py']:
        assert os.path.exists(p), f'STAGING FAILED: {p}'
        ast.parse(open(p).read())
    has_w = os.path.exists(f'{base}/agents/templates/pretrained_weights.pt')
    n_sol = len([f for f in os.listdir(f'{base}/agents/templates/solutions')]) if os.path.isdir(f'{base}/agents/templates/solutions') else 0
    print(f'rerun staging verified: my_agent + forge_agent in place; prior={"loaded" if has_w else "COLD"}; memory={n_sol} games')

    with open(f'{base}/agents/__init__.py', 'w') as f:
        f.write('''from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}
''')
    with open(f'{base}/.env', 'w') as f:
        f.write('''SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
RECORDINGS_DIR=/kaggle/working/server_recording
''')

    # MEMORY-FIRST (recall then solve): cache leads, live BFS for misses/stale.
    # STORE off (no rewrite during scoring).
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg PYTHONUNBUFFERED=1 \
        V19_CACHE_FIRST=1 V19_STORE_SOLUTIONS=0 V13_BFS_TIMEOUT=180 \
        python main.py --agent myagent 2>&1 | tee /kaggle/working/v19_run.log

**What to look for in the Logs tab (and `v19_run.log`):**
- `MEMORY HIT: level N solved from cache (K actions) — no live search needed` — the
  memory sweep recognised the level and replayed a verified learned plan. **Fast,
  zero search.** Most cached games/levels should be HITs.
- `MEMORY MISS: no cached solution for level N -> live BFS` — memory didn't cover
  this level, so the full v13/v17 ladder runs live (the genuine fallback).
- `MEMORY STALE: cached level N failed replay-verify -> falling through to live BFS`
  — a learned plan no longer wins (e.g. game-version drift); it is discarded and the
  agent solves live instead. Stale lines should be rare.
- `BFS ACTIVE: loaded <Class> from <path> ...` — the white-box source is reachable
  (needed for both the verify step and live fallthrough). If you instead see
  `[v19] no white-box source -> black-box fallback`, the source glob missed.

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(data=[['1_0', '1', True, 1]],
                              columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)

This is a dummy submission fallback, important to keep.

---

## How to read whether memory-first is working
1. **Submit** (Save & Run All → it runs during the scoring rerun).
2. **Memory should lead.** Expect many `MEMORY HIT` lines for the games already in
   `solutions/`, each clearing a level with zero search. This is the speed win of
   recalling learned solutions first.
3. **Live BFS covers the rest.** `MEMORY MISS` / `MEMORY STALE` → live BFS for that
   level. Held-out scored games you've never solved will be mostly misses and run
   live — that's expected and correct; only live BFS generalises to unseen games.
4. **Confirm the source is reachable.** A `BFS ACTIVE` line per game means verify +
   live fallthrough both work. Without it, verify can't run and memory-first degrades.

**Honesty note.** Every replayed plan is replay-verified to actually win the level on
a fresh engine before use, the cache only contains levels the agent solved itself in
earlier runs, and any uncached level is solved live. This is reuse of *your own
verified* prior work — recall first, genuine search second — not a stored answer-book.
Compare the BFS-first sibling `v19-to-kaggle` (cache as timeout backstop only); pick
the variant whose policy you want to submit under.